In [ ]:
%load_ext autoreload
%autoreload 2
import warnings

import os
import time
import pyproj
import numpy as np
import xarray as xr
import rioxarray
import s1_ard_core.computing.computation
import s1_ard_core.computing.coreg
import s1_ard_core.computing.dem
import s1_ard_core.computing.etad
import s1_ard_core.computing.gslc
import s1_ard_core.computing.io
import s1_ard_core.computing.orbit
import s1_ard_core.computing.shiroma
import s1_ard_core.computing.simul
from s1_ard_core.computing.io import read, write, normpath
import s1_ard_core.computing.slc

from s1_ard_core.computing.logger import logger
from pyproj.transformer import TransformerGroup

os.environ["PROJ_DATA"] = pyproj.datadir.get_data_dir()
warnings.filterwarnings("ignore")

In [ ]:
kml_path = "/home/morea/SENTINEL1/KML/S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml"
in_files = [
    "/home/morea/SENTINEL1/SALZBURG/44/SAFE/S1A_IW_SLC__1SDV_20241015T165929_20241015T165956_056116_06DDD0_2FE3.SAFE",
    "/home/morea/SENTINEL1/SALZBURG/44/SAFE/S1A_IW_SLC__1SDV_20241027T165929_20241027T165956_056291_06E4C5_095D.SAFE",
]
etad_files = None

path = "/home/morea/SENTINEL1/SALZBURG/44/S1-RP/ESD"
slcs_path = f"{path}/slcs"
wdir_path_static = f"{path}/STATIC"
wdir_path_tmp = f"{path}/DYNAMIC"

orbit_path = None
reference_date = "2024-10-15"
chunks_sr = {"acquisition_date": 1, "polarization": 1, "azimuth_time": 2048, "slant_range_time": 2048}
chunks = {"x": 2048, "y": 2048}
chunk_length = max([v for v in chunks.values()])
spacing = {"x": 30, "y": 30}
coreg = True
gamma_flattening = True
crs = None
recompute = False
multiburst = True

In [ ]:
type, n_jobs, job_memory = ["LOCAL", 12, "16GB"]
client = s1_ard_core.computing.computation.create_client(type, n_jobs, job_memory)
client

In [ ]:
polarization = ["V:H", "V:V"]
slcs = s1_ard_core.computing.slc.import_slc_stack(
    in_files,
    slcs_path,
    deramp=True,
    multiburst=multiburst,
    noise_removal_flag=True,
    apply_invalid_mask_flag=True,
    calib_type="BetaZero",
    chunks=chunks_sr,
)

In [ ]:
slcs = read(slcs_path, consolidate_metadata=False)

In [ ]:
bbox = s1_ard_core.computing.slc.get_bbox(slcs)
multiburst = True if "bursts" in slcs else False
acquisition_dates = s1_ard_core.computing.slc.extract_acquisition_dates(slcs, multiburst=multiburst)
if orbit_path is not None:
    slcs = s1_ard_core.computing.orbit.import_ORBIT(slcs, acquisition_dates, orbit_path, multiburst=multiburst)
if etad_files is not None:
    slcs = s1_ard_core.computing.etad.import_ETAD(
        slcs, acquisition_dates, etad_files, multiburst=multiburst, wdir_path=wdir_path_tmp
    )

In [ ]:
dem = s1_ard_core.computing.dem.import_dem(
    bbox=bbox,
    working_dir=wdir_path_tmp,
    chunks=chunks,
)

In [ ]:
dem_tree = s1_ard_core.computing.dem.stack_dem(dem, slcs, multiburst, spacing, wdir_path_tmp, chunks, crs=crs)

In [ ]:
# Checkpoint: Restart execution from here if the program crashes.

dem_tree = xr.DataTree()
for k, v in (slcs.bursts if multiburst else {"data": slcs}).items():
    path = f"bursts/{k}" if multiburst else "data"
    d = s1_ard_core.computing.io.read(f"{wdir_path_tmp}/{path}/dem_resampled", consolidate_metadata=False)
    if multiburst:
        dem_tree[path] = d
    else:
        dem_tree = xr.DataTree(d)

In [ ]:
orbits = s1_ard_core.computing.slc.extract_orbits(
    slcs,
    multiburst=multiburst,
    acquisition_date=[reference_date],
)

In [ ]:
simulation_tree = s1_ard_core.computing.simul.stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    chunks={**chunks, "acquisition_date": 1},
    master_image=True,
    recompute=recompute,
)

In [ ]:
simulation_ml_tree = s1_ard_core.computing.simul.stack_simulation_ml(
    simulation_tree=simulation_tree,
    reference_date=reference_date,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    recompute=recompute,
)

In [ ]:
orbits = s1_ard_core.computing.slc.extract_orbits(
    slcs,
    multiburst=multiburst,
)
orbits = orbits.where(orbits.acquisition_date != reference_date, drop=True)

In [ ]:
simulation_secondary_tree = s1_ard_core.computing.simul.stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_tmp,
    chunks={**chunks, "acquisition_date": 1},
    master_image=False,
    recompute=recompute,
)

In [ ]:
# Checkpoint: Restart execution from here if the program crashes.

simulation_secondary_tree = xr.DataTree()
for k in simulation_tree.bursts if multiburst else {"data": simulation_tree}:
    path = f"bursts/{k}" if multiburst else "data"
    data_path = s1_ard_core.computing.io.normpath(f"{wdir_path_tmp}/{path}/simulation")
    if multiburst:
        simulation_secondary_tree[path] = s1_ard_core.computing.io.read(data_path, consolidate_metadata=False)
    else:
        simulation_secondary_tree = s1_ard_core.computing.io.read(data_path, consolidate_metadata=False)

In [ ]:
slcs = s1_ard_core.computing.coreg.stack_coregistration(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    reference_date,
    simulation_secondary_tree=simulation_secondary_tree,
    multiburst=multiburst,
    working_directory=wdir_path_tmp,
    recompute=True,
)

In [ ]:
slcs

In [ ]:
# Checkpoint: Restart execution from here if the program crashes.

files = s1_ard_core.computing.io.glob(f"{wdir_path_tmp}/bursts/iw*")
for f in files:
    k = f.split("/")[-1]
    print(f)
    slcs[f"bursts/{k}/coreg"] = s1_ard_core.computing.io.read(f"{f}/coreg", consolidate_metadata=True)